# Fine-Tuning PhoBERT on Stratified 70/15/15 Merged Emotion Dataset

This notebook loads pre-split **Train (70% - 7,088 samples)**, **Validation (15% - 1,519 samples)**, and **Test (15% - 1,520 samples)** datasets generated from the merged VSMEC + GoEmotions corpus. It performs fine-tuning with Early Stopping and evaluates directly on the test set.

In [ ]:
# Step 1: Install required libraries
!pip install -q transformers datasets torch accelerate scikit-learn matplotlib


In [ ]:
# Step 2: Load train, val, test datasets
import json
import numpy as np

with open("phobert_train.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open("phobert_val.json", "r", encoding="utf-8") as f:
    val_data = json.load(f)
with open("phobert_test.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Train samples: {len(train_data)}")
print(f"Val samples  : {len(val_data)}")
print(f"Test samples : {len(test_data)}")


In [ ]:
# Step 3: Tokenization & HuggingFace Dataset Conversion
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import classification_report, accuracy_score, f1_score
import torch

model_name = "vinai/phobert-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

train_ds = Dataset.from_dict({"text": [x["text"] for x in train_data], "label": [x["label"] for x in train_data]}).map(tokenize_fn, batched=True)
val_ds = Dataset.from_dict({"text": [x["text"] for x in val_data], "label": [x["label"] for x in val_data]}).map(tokenize_fn, batched=True)
test_ds = Dataset.from_dict({"text": [x["text"] for x in test_data], "label": [x["label"] for x in test_data]}).map(tokenize_fn, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=7)


In [ ]:
# Step 4: Fine-Tuning Setup for PhoBERT-Large (Standard CrossEntropy & Cosine Decay)
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
import numpy as np

training_args = TrainingArguments(
    output_dir="./results_phobert_large_v3",
    num_train_epochs=12,                # Trần 12 epochs giúp Cosine Scheduler duỗi mịn chạm trần F1
    per_device_train_batch_size=8,      # Batch size 8 vừa VRAM T4 GPU (16GB)
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,       # Tích lũy gradient (Effective Batch Size = 16)
    learning_rate=8e-6,                 # LR nhỏ mịn (8e-6) tối ưu riêng cho PhoBERT-Large
    warmup_steps=300,                   # 300 steps warmup tránh shock gradient
    lr_scheduler_type="cosine",          # Cosine decay giúp hội tụ sâu và mịn
    weight_decay=0.05,                  # L2 Regularization mạnh chống Overfit
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True,                          # Mixed precision Float16
    logging_steps=50
)

def compute_metrics(eval_pred):
    logits, l_labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(l_labels, preds)
    macro_f1 = f1_score(l_labels, preds, average="macro")
    return {"accuracy": acc, "f1": macro_f1}

# Trainer chuẩn HuggingFace (Học tự nhiên không gượng ép phạt trọng số)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # Tự ngắt nếu 2 epoch F1 không tăng
)

print("Starting PhoBERT-Large Fine-Tuning (V3 Perfect Setup)...")
trainer.train()

# Save Best Model
model.save_pretrained("./phobert_emotion_final")
tokenizer.save_pretrained("./phobert_emotion_final")
print("Best model saved to ./phobert_emotion_final")


In [ ]:
# Step 5: Final Evaluation on Independent Test Set & Export Report
print("\n==================================================")
print("   EVALUATING BEST MODEL ON HELD-OUT TEST SET")
print("==================================================")

test_results = trainer.predict(test_ds)
test_preds = np.argmax(test_results.predictions, axis=1)
test_labels = [x["label"] for x in test_data]
label_names = ["Enjoyment", "Sadness", "Disgust", "Anger", "Fear", "Surprise", "Other"]

report_str = classification_report(test_labels, test_preds, target_names=label_names, digits=4)
print(report_str)

# Save evaluation report to text file
with open("test_log.txt", "w", encoding="utf-8") as f:
    f.write(report_str)


In [ ]:
# Step 6: Plot Training Curves & Auto Download Package
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))
history = trainer.state.log_history
epochs = []
val_f1s = []
train_losses = []
val_losses = []

for log in history:
    if "eval_f1" in log:
        epochs.append(log["epoch"])
        val_f1s.append(log["eval_f1"] * 100)
        val_losses.append(log["eval_loss"])
    elif "loss" in log:
        train_losses.append(log["loss"])

plt.subplot(1, 2, 1)
if len(epochs) > 0 and len(val_f1s) == len(epochs):
    plt.plot(epochs, val_f1s, "o-", color="tab:blue", linewidth=2, label="Validation Macro F1")
plt.title("Validation Macro F1-Score per Epoch", fontsize=12, fontweight="bold")
plt.xlabel("Epoch", fontsize=10)
plt.ylabel("Macro F1 (%)", fontsize=10)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.subplot(1, 2, 2)
if len(epochs) > 0 and len(val_losses) == len(epochs):
    plt.plot(epochs, val_losses, "o-", color="tab:red", linewidth=2, label="Validation Loss")
plt.title("Training Convergence & Validation Loss", fontsize=12, fontweight="bold")
plt.xlabel("Epoch", fontsize=10)
plt.ylabel("Loss", fontsize=10)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig("training_performance_curves.png", dpi=300)
plt.show()
print("Learning curves plot saved to training_performance_curves.png!")

# Zip final model, test report & learning curve plot
!zip -r phobert_emotion_final.zip ./phobert_emotion_final test_log.txt training_performance_curves.png

# Auto download file zip to local machine
try:
    from google.colab import files
    files.download("phobert_emotion_final.zip")
    print("Auto-downloading phobert_emotion_final.zip (with model + log + plot curves) to your browser...")
except Exception as e:
    print("Not running in Colab environment, skip auto-download.")
